# Model Metadata and Versioning — Hands-On Application

## Objective

Master ONNX model metadata management: annotating models with producer info, version tracking,
opset manipulation, custom metadata properties for ML-ops, and building version comparison tools.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports and model factory |
| 2 | [Exercise 1: Fully Annotated Models](#2-exercise-1) | Set all metadata fields |
| 3 | [Exercise 2: Metadata Audit Tool](#3-exercise-2) | Inspect and report existing metadata |
| 4 | [Exercise 3: Opset Version Modification](#4-exercise-3) | Change opset on existing models |
| 5 | [Exercise 4: Save/Reload Metadata Round-Trip](#5-exercise-4) | Persistence verification |
| 6 | [Exercise 5: Multi-Domain Opset Imports](#6-exercise-5) | Standard + ML + custom domains |
| 7 | [Exercise 6: Custom metadata_props for ML-Ops](#7-exercise-6) | Training lineage tracking |
| 8 | [Exercise 7: Version Comparison Tool](#8-exercise-7) | Diff two model versions |
| 9 | [Challenge: Model Versioning System](#9-challenge) | Full lifecycle management |
| 10 | [Summary](#10-summary) | Skills review |

In [ ]:
# 1. Setup <a id="1-setup"></a>
# !pip install onnx numpy matplotlib --quiet

import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs, version_converter
import numpy as np
import copy
import json
import tempfile
import os
import time
import hashlib
from datetime import datetime
import matplotlib.pyplot as plt

print(f"ONNX version: {onnx.__version__}")
print(f"IR version: {onnx.IR_VERSION}")
print(f"Default opset: {defs.onnx_opset_version()}")

In [ ]:
def make_test_model(opset: int = 17, hidden: int = 64) -> onnx.ModelProto:
    """Build a simple MLP for metadata exercises."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", hidden])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 32])
    W = numpy_helper.from_array(np.random.randn(hidden, 32).astype(np.float32) * 0.01, "W")
    b = numpy_helper.from_array(np.zeros(32, dtype=np.float32), "b")
    nodes = [
        helper.make_node("MatMul", ["X", "W"], ["mm"]),
        helper.make_node("Add", ["mm", "b"], ["z"]),
        helper.make_node("Relu", ["z"], ["Y"]),
    ]
    graph = helper.make_graph(nodes, "test_graph", [X], [Y], initializer=[W, b])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
    checker.check_model(model)
    return model


test_model = make_test_model()
print(f"Test model: {len(test_model.graph.node)} nodes, opset {test_model.opset_import[0].version}")

## 2. Exercise 1: Fully Annotated Models <a id="2-exercise-1"></a>

ONNX ModelProto has several metadata fields:

| Field | Type | Purpose |
|-------|------|--------|
| `producer_name` | string | Framework that created the model |
| `producer_version` | string | Framework version |
| `domain` | string | Organization/project domain |
| `model_version` | int64 | User-defined version number |
| `doc_string` | string | Human-readable description |
| `metadata_props` | map<string,string> | Arbitrary key-value pairs |

These fields carry no computational semantics but are essential for model governance.

In [ ]:
def build_annotated_model(
    producer: str,
    producer_version: str,
    domain: str,
    model_version: int,
    doc_string: str,
    metadata: dict,
    opset: int = 17,
) -> onnx.ModelProto:
    """Build a fully annotated ONNX model."""
    model = make_test_model(opset)

    model.producer_name = producer
    model.producer_version = producer_version
    model.domain = domain
    model.model_version = model_version
    model.doc_string = doc_string

    for key, value in metadata.items():
        entry = model.metadata_props.add()
        entry.key = key
        entry.value = str(value)

    checker.check_model(model)
    return model


# Create a production-quality annotated model
model = build_annotated_model(
    producer="PyTorch",
    producer_version="2.1.0",
    domain="com.example.nlp",
    model_version=3,
    doc_string="Sentiment classifier trained on IMDB reviews. Binary classification.",
    metadata={
        "accuracy": "0.923",
        "f1_score": "0.918",
        "training_dataset": "imdb_50k",
        "training_epochs": "25",
        "optimizer": "AdamW",
        "learning_rate": "3e-4",
        "export_date": "2024-06-15T10:30:00Z",
        "git_commit": "a1b2c3d",
        "license": "Apache-2.0",
    },
)

# Verify all fields are set
assert model.producer_name == "PyTorch"
assert model.model_version == 3
assert model.domain == "com.example.nlp"
assert len(model.metadata_props) == 9

print("Model Annotation Summary:")
print(f"  Producer:    {model.producer_name} v{model.producer_version}")
print(f"  Domain:      {model.domain}")
print(f"  Version:     {model.model_version}")
print(f"  Doc:         {model.doc_string[:60]}...")
print(f"  Metadata:    {len(model.metadata_props)} entries")
for prop in model.metadata_props:
    print(f"    {prop.key}: {prop.value}")
print("\nAll annotations set correctly. ✓")

## 3. Exercise 2: Metadata Audit Tool <a id="3-exercise-2"></a>

Build a comprehensive audit function that extracts and validates all metadata from an ONNX model.

**Audit checks:**
- Required fields populated (producer_name, model_version)
- IR version compatibility
- OpSet recency: is the declared opset within $n$ versions of the latest?

$$\text{opset\_lag} = v_{\text{latest}} - v_{\text{model}}$$

In [ ]:
def audit_model_metadata(model: onnx.ModelProto) -> dict:
    """Comprehensive metadata audit with quality scoring."""
    latest_opset = defs.onnx_opset_version()
    report = {
        "ir_version": model.ir_version,
        "opset_imports": [(oi.domain or "(default)", oi.version) for oi in model.opset_import],
        "producer_name": model.producer_name or "(empty)",
        "producer_version": model.producer_version or "(empty)",
        "domain": model.domain or "(empty)",
        "model_version": model.model_version,
        "doc_string": model.doc_string[:100] if model.doc_string else "(empty)",
        "metadata_count": len(model.metadata_props),
        "metadata": {p.key: p.value for p in model.metadata_props},
    }

    # Quality checks
    checks = []
    score = 0
    max_score = 8

    if model.producer_name:
        checks.append(("✓", "producer_name set")); score += 1
    else:
        checks.append(("✗", "producer_name missing"))

    if model.producer_version:
        checks.append(("✓", "producer_version set")); score += 1
    else:
        checks.append(("⚠", "producer_version empty"))

    if model.domain:
        checks.append(("✓", "domain set")); score += 1
    else:
        checks.append(("⚠", "domain empty"))

    if model.model_version > 0:
        checks.append(("✓", f"model_version = {model.model_version}")); score += 1
    else:
        checks.append(("✗", "model_version not set (0)"))

    if model.doc_string:
        checks.append(("✓", "doc_string present")); score += 1
    else:
        checks.append(("⚠", "doc_string missing"))

    declared_opset = model.opset_import[0].version
    opset_lag = latest_opset - declared_opset
    if opset_lag <= 3:
        checks.append(("✓", f"opset {declared_opset} (lag={opset_lag}, current)"))
        score += 1
    elif opset_lag <= 6:
        checks.append(("⚠", f"opset {declared_opset} (lag={opset_lag}, aging)"))
    else:
        checks.append(("✗", f"opset {declared_opset} (lag={opset_lag}, outdated)"))

    if len(model.metadata_props) >= 3:
        checks.append(("✓", f"{len(model.metadata_props)} metadata entries")); score += 1
    elif len(model.metadata_props) > 0:
        checks.append(("⚠", f"only {len(model.metadata_props)} metadata entries"))
    else:
        checks.append(("✗", "no metadata_props"))

    try:
        checker.check_model(model)
        checks.append(("✓", "passes onnx.checker")); score += 1
    except Exception as e:
        checks.append(("✗", f"checker failed: {str(e)[:50]}"))

    report["checks"] = checks
    report["quality_score"] = f"{score}/{max_score}"
    report["opset_lag"] = opset_lag

    # Print report
    print("╔══════════════════════════════════════════════════╗")
    print("║        MODEL METADATA AUDIT REPORT               ║")
    print("╠══════════════════════════════════════════════════╣")
    print(f"║  Quality Score: {report['quality_score']:>34}║")
    print(f"║  IR Version:    {report['ir_version']:<34}║")
    for d, v in report["opset_imports"]:
        print(f"║  OpSet:         {d} v{v:<27}║")
    print("╠══════════════════════════════════════════════════╣")
    for status, msg in checks:
        print(f"║  {status} {msg:<46}║")
    print("╚══════════════════════════════════════════════════╝")

    return report


# Test with our annotated model
print("=== Well-annotated model ===")
report1 = audit_model_metadata(model)

# Test with a bare model
print("\n=== Bare model (no annotations) ===")
bare = make_test_model(13)
report2 = audit_model_metadata(bare)

## 4. Exercise 3: Opset Version Modification <a id="4-exercise-3"></a>

Modify the opset version on an existing model. This involves:
1. Changing `opset_import` declarations
2. Using `version_converter.convert_version()` for safe upgrades
3. Verifying the model still validates

The conversion process applies adapter transformations:
$$G_{v_s} \xrightarrow{A_{v_s \to v_{s+1}}} G_{v_{s+1}} \xrightarrow{\cdots} G_{v_t}$$

In [ ]:
def modify_opset(model: onnx.ModelProto, target_opset: int,
                 use_converter: bool = True) -> tuple:
    """Modify model opset version and validate."""
    original_opset = model.opset_import[0].version
    result = {"source": original_opset, "target": target_opset}

    if use_converter:
        try:
            converted = version_converter.convert_version(model, target_opset)
            checker.check_model(converted)
            result["status"] = "converted"
            result["nodes_before"] = len(model.graph.node)
            result["nodes_after"] = len(converted.graph.node)
            return converted, result
        except Exception as e:
            result["status"] = "failed"
            result["error"] = str(e)[:80]
            return None, result
    else:
        # Manual opset change (unsafe — only changes declaration)
        modified = copy.deepcopy(model)
        modified.opset_import[0].version = target_opset
        try:
            checker.check_model(modified)
            result["status"] = "manual_ok"
        except Exception as e:
            result["status"] = "manual_invalid"
            result["error"] = str(e)[:80]
        return modified, result


# Test conversion across multiple opset targets
base_model = make_test_model(opset=13)
targets = [11, 13, 15, 17, 18, 19]

print(f"Source model: opset {base_model.opset_import[0].version}")
print(f"\n{'Target':>7} | {'Status':<15} | {'Nodes':>12} | Notes")
print("-" * 60)

for target in targets:
    converted, result = modify_opset(base_model, target)
    node_info = f"{result.get('nodes_before', '?')} → {result.get('nodes_after', '?')}"
    notes = result.get("error", "success")
    print(f"{target:>7} | {result['status']:<15} | {node_info:>12} | {notes}")

    if converted is not None:
        actual = converted.opset_import[0].version
        assert actual == target, f"Expected opset {target}, got {actual}"

print("\nAll conversions completed. ✓")

## 5. Exercise 4: Save/Reload Metadata Round-Trip <a id="5-exercise-4"></a>

Verify that all metadata fields survive serialization → deserialization.

This tests that:
- `producer_name`, `producer_version`, `domain` persist
- `model_version` (int64) persists
- `doc_string` (including special characters) persists
- All `metadata_props` key-value pairs persist in order

In [ ]:
def verify_metadata_roundtrip(model: onnx.ModelProto) -> bool:
    """Verify all metadata survives save/load cycle."""
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "metadata_test.onnx")
        onnx.save(model, path)
        loaded = onnx.load(path)

    checks = []

    # Core fields
    checks.append(("producer_name", model.producer_name == loaded.producer_name))
    checks.append(("producer_version", model.producer_version == loaded.producer_version))
    checks.append(("domain", model.domain == loaded.domain))
    checks.append(("model_version", model.model_version == loaded.model_version))
    checks.append(("doc_string", model.doc_string == loaded.doc_string))
    checks.append(("ir_version", model.ir_version == loaded.ir_version))

    # Opset imports
    orig_opsets = [(oi.domain, oi.version) for oi in model.opset_import]
    load_opsets = [(oi.domain, oi.version) for oi in loaded.opset_import]
    checks.append(("opset_imports", orig_opsets == load_opsets))

    # Metadata props
    orig_meta = {p.key: p.value for p in model.metadata_props}
    load_meta = {p.key: p.value for p in loaded.metadata_props}
    checks.append(("metadata_props", orig_meta == load_meta))

    # Print results
    all_pass = all(ok for _, ok in checks)
    for field, ok in checks:
        status = "✓" if ok else "✗"
        print(f"  {status} {field}")

    return all_pass


# Test 1: Rich metadata model
print("Test 1: Fully annotated model")
rich = build_annotated_model(
    producer="TensorFlow",
    producer_version="2.15.0",
    domain="org.tensorflow.models",
    model_version=42,
    doc_string="Multi-class classifier with special chars: é, ñ, ü, 中文",
    metadata={"key_with_dots": "val.ue", "empty_val": "", "long_val": "x" * 1000},
)
assert verify_metadata_roundtrip(rich), "Rich model failed"

# Test 2: Model with empty/default metadata
print("\nTest 2: Minimal metadata")
minimal = make_test_model()
minimal.producer_name = "min"
assert verify_metadata_roundtrip(minimal), "Minimal model failed"

# Test 3: Bytes round-trip
print("\nTest 3: Bytes serialization")
raw = rich.SerializeToString()
reloaded = onnx.load_model_from_string(raw)
assert reloaded.producer_name == "TensorFlow"
assert reloaded.model_version == 42
assert {p.key: p.value for p in reloaded.metadata_props} == {p.key: p.value for p in rich.metadata_props}
print("  ✓ Bytes round-trip preserves all metadata")

print("\nAll round-trip tests passed. ✓")

## 6. Exercise 5: Multi-Domain Opset Imports <a id="6-exercise-5"></a>

ONNX supports multiple opset domains in a single model:
- `""` (empty string) — default ONNX domain
- `"ai.onnx.ml"` — ML-specific operators (tree ensembles, SVMs, etc.)
- Custom domains — user-defined operator sets

Each domain has independent versioning:
$$\text{opset\_imports} = \{(d_1, v_1), (d_2, v_2), \ldots, (d_n, v_n)\}$$

In [ ]:
# Build a model with multiple opset domains
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 64])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 32])

W = numpy_helper.from_array(np.random.randn(64, 32).astype(np.float32) * 0.01, "W")
b = numpy_helper.from_array(np.zeros(32, dtype=np.float32), "b")

nodes = [
    helper.make_node("MatMul", ["X", "W"], ["mm"]),
    helper.make_node("Add", ["mm", "b"], ["z"]),
    helper.make_node("Relu", ["z"], ["Y"]),
]

graph = helper.make_graph(nodes, "multi_domain", [X], [Y], initializer=[W, b])

# Multiple opset imports
multi_domain_model = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 17),            # Standard ONNX ops
        helper.make_opsetid("ai.onnx.ml", 3),   # ML domain
        helper.make_opsetid("com.myorg.custom", 1),  # Custom domain
    ],
)

# Inspect opset imports
print("Multi-Domain Opset Imports:")
print(f"{'Domain':<25} | {'Version':>7}")
print("-" * 40)
for oi in multi_domain_model.opset_import:
    domain = oi.domain or "(default/ai.onnx)"
    print(f"{domain:<25} | {oi.version:>7}")

# Verify structure
assert len(multi_domain_model.opset_import) == 3
domains = {oi.domain: oi.version for oi in multi_domain_model.opset_import}
assert domains[""] == 17
assert domains["ai.onnx.ml"] == 3
assert domains["com.myorg.custom"] == 1

# Round-trip test
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "multi_domain.onnx")
    onnx.save(multi_domain_model, path)
    loaded = onnx.load(path)
    loaded_domains = {oi.domain: oi.version for oi in loaded.opset_import}
    assert loaded_domains == domains
    print("\nMulti-domain opsets survive round-trip. ✓")

# Programmatic domain manipulation
def add_opset_domain(model: onnx.ModelProto, domain: str, version: int):
    """Add or update an opset domain import."""
    for oi in model.opset_import:
        if oi.domain == domain:
            oi.version = version
            return
    new_oi = model.opset_import.add()
    new_oi.domain = domain
    new_oi.version = version

test = make_test_model()
add_opset_domain(test, "ai.onnx.ml", 3)
add_opset_domain(test, "com.example", 2)
print(f"\nAfter adding domains: {[(oi.domain or 'default', oi.version) for oi in test.opset_import]}")

## 7. Exercise 6: Custom metadata_props for ML-Ops <a id="7-exercise-6"></a>

Build a metadata manager class for production ML-ops workflows. Track:
- Training lineage (dataset, hyperparameters, hardware)
- Performance metrics (accuracy, latency)
- Deployment info (target platforms, quantization status)

The `metadata_props` field is a repeated `StringStringEntryProto` — essentially a list of (key, value) pairs.

In [ ]:
class MLOpsMetadata:
    """ML-Ops metadata manager for ONNX models."""

    PREFIXES = {
        "training": "train.",
        "metrics": "metrics.",
        "deployment": "deploy.",
        "data": "data.",
        "system": "sys.",
    }

    def __init__(self, model: onnx.ModelProto):
        self.model = model

    def set(self, key: str, value: str) -> None:
        for prop in self.model.metadata_props:
            if prop.key == key:
                prop.value = value
                return
        entry = self.model.metadata_props.add()
        entry.key = key
        entry.value = value

    def get(self, key: str, default: str = None) -> str:
        for prop in self.model.metadata_props:
            if prop.key == key:
                return prop.value
        return default

    def remove(self, key: str) -> bool:
        for i, prop in enumerate(self.model.metadata_props):
            if prop.key == key:
                del self.model.metadata_props[i]
                return True
        return False

    def set_training_info(self, **kwargs):
        for k, v in kwargs.items():
            self.set(f"{self.PREFIXES['training']}{k}", str(v))

    def set_metrics(self, **kwargs):
        for k, v in kwargs.items():
            self.set(f"{self.PREFIXES['metrics']}{k}", str(v))

    def set_deployment_info(self, **kwargs):
        for k, v in kwargs.items():
            self.set(f"{self.PREFIXES['deployment']}{k}", str(v))

    def get_by_prefix(self, prefix: str) -> dict:
        return {
            p.key: p.value for p in self.model.metadata_props
            if p.key.startswith(prefix)
        }

    def to_dict(self) -> dict:
        return {p.key: p.value for p in self.model.metadata_props}

    def summary(self):
        all_meta = self.to_dict()
        categories = {}
        for key, val in all_meta.items():
            prefix = key.split(".")[0] + "." if "." in key else "(other)"
            categories.setdefault(prefix, []).append((key, val))

        print(f"\nML-Ops Metadata Summary ({len(all_meta)} entries):")
        for cat, items in sorted(categories.items()):
            print(f"\n  [{cat}]")
            for k, v in items:
                print(f"    {k}: {v}")


# Usage demonstration
model = make_test_model()
meta = MLOpsMetadata(model)

# Set training info
meta.set_training_info(
    epochs=50,
    batch_size=32,
    optimizer="AdamW",
    lr="3e-4",
    weight_decay=0.01,
    hardware="NVIDIA A100",
    duration_hours=2.5,
)

# Set metrics
meta.set_metrics(
    accuracy=0.934,
    f1_macro=0.921,
    inference_ms=1.2,
    memory_mb=45,
)

# Set deployment info
meta.set_deployment_info(
    target="onnxruntime-gpu",
    quantized="false",
    min_opset=13,
    platforms="linux-x64,linux-arm64",
)

meta.summary()

# Verify retrieval
assert meta.get("train.epochs") == "50"
assert meta.get("metrics.accuracy") == "0.934"
assert meta.get("nonexistent", "default") == "default"

# Get by category
training = meta.get_by_prefix("train.")
assert len(training) == 7
print(f"\nTraining metadata: {len(training)} entries ✓")

## 8. Exercise 7: Version Comparison Tool <a id="8-exercise-7"></a>

Build a tool that compares two model versions side-by-side, highlighting changes in:
- Architecture (nodes, ops)
- Parameters (count, total size)
- Metadata (new/modified/removed entries)
- Performance metrics (if tracked in metadata)

In [ ]:
def compare_models(m1: onnx.ModelProto, m2: onnx.ModelProto,
                   label1: str = "v1", label2: str = "v2") -> dict:
    """Compare two model versions in detail."""
    def model_info(m):
        return {
            "opset": m.opset_import[0].version,
            "ir_version": m.ir_version,
            "n_nodes": len(m.graph.node),
            "n_params": sum(int(np.prod(list(i.dims))) for i in m.graph.initializer),
            "size_bytes": len(m.SerializeToString()),
            "ops": sorted(set(n.op_type for n in m.graph.node)),
            "producer": m.producer_name,
            "version": m.model_version,
            "metadata": {p.key: p.value for p in m.metadata_props},
        }

    info1 = model_info(m1)
    info2 = model_info(m2)

    # Metadata diff
    meta_keys_1 = set(info1["metadata"].keys())
    meta_keys_2 = set(info2["metadata"].keys())
    added_keys = meta_keys_2 - meta_keys_1
    removed_keys = meta_keys_1 - meta_keys_2
    common_keys = meta_keys_1 & meta_keys_2
    modified_keys = {k for k in common_keys if info1["metadata"][k] != info2["metadata"][k]}

    diff = {
        "node_delta": info2["n_nodes"] - info1["n_nodes"],
        "param_delta": info2["n_params"] - info1["n_params"],
        "size_delta": info2["size_bytes"] - info1["size_bytes"],
        "opset_change": (info1["opset"], info2["opset"]),
        "ops_added": set(info2["ops"]) - set(info1["ops"]),
        "ops_removed": set(info1["ops"]) - set(info2["ops"]),
        "meta_added": added_keys,
        "meta_removed": removed_keys,
        "meta_modified": modified_keys,
    }

    # Print comparison
    print(f"\n{'═'*60}")
    print(f"  MODEL COMPARISON: {label1} vs {label2}")
    print(f"{'═'*60}")
    print(f"\n  {'Metric':<20} | {label1:>12} | {label2:>12} | {'Delta':>12}")
    print(f"  {'-'*60}")
    print(f"  {'OpSet':<20} | {info1['opset']:>12} | {info2['opset']:>12} | {'':>12}")
    print(f"  {'Nodes':<20} | {info1['n_nodes']:>12} | {info2['n_nodes']:>12} | {diff['node_delta']:>+12}")
    print(f"  {'Parameters':<20} | {info1['n_params']:>12,} | {info2['n_params']:>12,} | {diff['param_delta']:>+12,}")
    print(f"  {'Size (bytes)':<20} | {info1['size_bytes']:>12,} | {info2['size_bytes']:>12,} | {diff['size_delta']:>+12,}")
    print(f"  {'Model Version':<20} | {info1['version']:>12} | {info2['version']:>12} | {'':>12}")

    if diff["ops_added"]:
        print(f"\n  New ops in {label2}: {diff['ops_added']}")
    if diff["ops_removed"]:
        print(f"  Removed ops: {diff['ops_removed']}")

    if diff["meta_added"] or diff["meta_removed"] or diff["meta_modified"]:
        print(f"\n  Metadata changes:")
        for k in diff["meta_added"]:
            print(f"    + {k} = {info2['metadata'][k]}")
        for k in diff["meta_removed"]:
            print(f"    - {k} (was: {info1['metadata'][k]})")
        for k in diff["meta_modified"]:
            print(f"    ~ {k}: {info1['metadata'][k]} → {info2['metadata'][k]}")

    return diff


# Create two model versions with differences
v1 = make_test_model(opset=13, hidden=64)
v1.model_version = 1
v1.producer_name = "pytorch"
for k, v in {"accuracy": "0.85", "dataset": "train_v1"}.items():
    e = v1.metadata_props.add(); e.key, e.value = k, v

v2 = make_test_model(opset=17, hidden=64)
v2.model_version = 2
v2.producer_name = "pytorch"
for k, v in {"accuracy": "0.92", "dataset": "train_v2", "quantized": "int8"}.items():
    e = v2.metadata_props.add(); e.key, e.value = k, v

diff = compare_models(v1, v2, "v1 (opset 13)", "v2 (opset 17)")

# Assertions
assert diff["opset_change"] == (13, 17)
assert "quantized" in diff["meta_added"]
assert "accuracy" in diff["meta_modified"]
print("\nComparison verified. ✓")

## 9. Challenge: Model Versioning System <a id="9-challenge"></a>

Build a complete model versioning system that:
1. Registers model versions with full metadata
2. Tracks evolution over time (accuracy improvement)
3. Supports rollback to previous versions
4. Generates visual version history
5. Enforces metadata requirements (schema validation)

In [ ]:
class ModelVersioningSystem:
    """Full model versioning system with lifecycle management."""

    REQUIRED_METADATA = ["accuracy", "training_epochs"]

    def __init__(self, base_dir: str):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)
        self.history = {}  # model_name -> [{version, path, metadata, timestamp, checksum}]

    def validate_metadata(self, metadata: dict) -> list:
        """Check required metadata keys are present."""
        missing = [k for k in self.REQUIRED_METADATA if k not in metadata]
        return missing

    def register(self, name: str, model: onnx.ModelProto, metadata: dict) -> dict:
        """Register a new version."""
        missing = self.validate_metadata(metadata)
        if missing:
            raise ValueError(f"Missing required metadata: {missing}")

        if name not in self.history:
            self.history[name] = []

        version = len(self.history[name]) + 1
        model.model_version = version

        # Set metadata
        while model.metadata_props:
            model.metadata_props.pop()
        for k, v in metadata.items():
            e = model.metadata_props.add()
            e.key, e.value = k, str(v)

        # Save
        fname = f"{name}_v{version}.onnx"
        path = os.path.join(self.base_dir, fname)
        onnx.save(model, path)

        raw = model.SerializeToString()
        entry = {
            "version": version,
            "path": path,
            "metadata": metadata,
            "timestamp": datetime.now().isoformat(),
            "checksum": hashlib.sha256(raw).hexdigest()[:16],
            "size": os.path.getsize(path),
            "n_params": sum(int(np.prod(list(i.dims))) for i in model.graph.initializer),
        }
        self.history[name].append(entry)
        return entry

    def load_version(self, name: str, version: int = None) -> onnx.ModelProto:
        """Load a specific version (default=latest)."""
        versions = self.history.get(name, [])
        if not versions:
            raise KeyError(f"No model '{name}' registered")

        entry = versions[-1] if version is None else versions[version - 1]
        model = onnx.load(entry["path"])

        actual_hash = hashlib.sha256(model.SerializeToString()).hexdigest()[:16]
        assert actual_hash == entry["checksum"], "Integrity check failed!"
        return model

    def rollback(self, name: str, to_version: int) -> onnx.ModelProto:
        """Rollback: mark the target version as the active latest."""
        model = self.load_version(name, to_version)
        print(f"Rolled back '{name}' to v{to_version}")
        return model

    def get_metric_history(self, name: str, metric_key: str) -> list:
        """Get the history of a specific metric across versions."""
        return [
            (e["version"], float(e["metadata"].get(metric_key, 0)))
            for e in self.history.get(name, [])
        ]

    def visualize_history(self, name: str, metric: str = "accuracy"):
        """Plot version history for a metric."""
        history = self.get_metric_history(name, metric)
        if not history:
            print("No history to plot.")
            return

        versions, values = zip(*history)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Metric over versions
        axes[0].plot(versions, values, "bo-", markersize=8, linewidth=2)
        axes[0].set_xlabel("Version")
        axes[0].set_ylabel(metric.capitalize())
        axes[0].set_title(f"{name}: {metric} over versions", fontweight="bold")
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xticks(list(versions))

        # Model size over versions
        sizes = [e["size"] / 1024 for e in self.history[name]]
        axes[1].bar(versions, sizes, color="#4CAF50", alpha=0.8)
        axes[1].set_xlabel("Version")
        axes[1].set_ylabel("Size (KB)")
        axes[1].set_title(f"{name}: model size", fontweight="bold")
        axes[1].set_xticks(list(versions))
        axes[1].grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.show()


# Simulate a model development lifecycle
with tempfile.TemporaryDirectory(prefix="versioning_") as tmpdir:
    vcs = ModelVersioningSystem(tmpdir)

    # Register multiple versions with improving accuracy
    training_runs = [
        {"accuracy": "0.78", "training_epochs": "5", "lr": "1e-3", "hidden": "64"},
        {"accuracy": "0.85", "training_epochs": "15", "lr": "5e-4", "hidden": "128"},
        {"accuracy": "0.91", "training_epochs": "30", "lr": "3e-4", "hidden": "128"},
        {"accuracy": "0.87", "training_epochs": "50", "lr": "1e-4", "hidden": "256"},  # overfit
        {"accuracy": "0.94", "training_epochs": "40", "lr": "3e-4", "hidden": "256"},
    ]

    for meta in training_runs:
        h = int(meta["hidden"])
        m = make_test_model(hidden=h)
        entry = vcs.register("classifier", m, meta)
        print(f"Registered v{entry['version']}: acc={meta['accuracy']}, "
              f"size={entry['size']:,}B, hash={entry['checksum']}")

    # Visualize improvement
    vcs.visualize_history("classifier", "accuracy")

    # Rollback to v3 (before overfit)
    rolled_back = vcs.rollback("classifier", 3)
    assert rolled_back.model_version == 3

    # Verify metadata validation
    try:
        vcs.register("bad", make_test_model(), {"only_accuracy": "0.9"})
        assert False, "Should have raised"
    except ValueError as e:
        print(f"\nMetadata validation works: {e} ✓")

## 10. Summary <a id="10-summary"></a>

| Exercise | Skill | Key Insight |
|----------|-------|---------|
| 1. Annotated Models | Set all metadata fields | Complete provenance tracking |
| 2. Audit Tool | Quality scoring | Enforce metadata standards |
| 3. Opset Modification | `version_converter` | Safe opset up/downgrade |
| 4. Round-Trip | Persistence verification | All fields survive serialization |
| 5. Multi-Domain | Multiple opset imports | Independent domain versioning |
| 6. ML-Ops Props | Structured metadata | Prefixed key-value taxonomy |
| 7. Version Comparison | Side-by-side diff | Architecture + metadata deltas |
| Challenge | Versioning System | Full lifecycle with rollback |

### Key Formulas

- Opset compatibility: $\forall d \in \text{domains}(M): v_M(d) \leq v_R(d)$
- Opset lag: $\text{lag} = v_{\text{latest}} - v_{\text{model}}$
- Version conversion chain: $G_{v_s} \xrightarrow{A} G_{v_{s+1}} \xrightarrow{A} \cdots \xrightarrow{A} G_{v_t}$